<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/13a_lstm_tuning_compacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB13a — Tuning Compacto da LSTM

## Papel do notebook no pipeline

O NB13a é um notebook intermediário derivado do NB13. Sua função é avaliar, de forma controlada e rastreável, se um tuning compacto de hiperparâmetros da LSTM altera a conclusão metodológica do NB13 sobre a comparação entre arquitetura recorrente e modelo tabular vencedor do NB11.

Por convenção metodológica dos notebooks intermediários, o NB13a pode ler artefatos de notebooks anteriores, incluindo o NB13, mas não deve sobrescrever nenhum arquivo produzido pelo notebook pai. Todos os artefatos gerados por esta etapa usam o prefixo `13a_`.

## Motivação

O NB13 avaliou uma LSTM simples como comparador sequencial não trivial, usando os artefatos materializados pelo NB11 e os conjuntos de features `raw_7` e `temporal_core`. A melhor configuração obteve F1 médio no TimeSeriesSplit próximo, porém inferior, ao modelo tabular vencedor do NB11.

A proximidade entre os resultados torna metodologicamente relevante verificar se uma varredura compacta de hiperparâmetros da LSTM poderia produzir ganho suficiente para alterar a decisão do pipeline. O NB13a responde a essa questão sem transformar a dissertação em uma busca ampla de arquitetura neural.

## Questão experimental

O tuning compacto da LSTM melhora o desempenho médio no TimeSeriesSplit de forma suficiente para substituir os escores do NB11 no NB14?

## Escopo controlado

Para manter o custo computacional sob controle e evitar garimpagem experimental, o NB13a concentra o tuning no ponto mais relevante da comparação:

```text
Cenário: W5_K24_H12_P1_TRAIN_M2S
Feature set: temporal_core
Protocolo principal: TimeSeriesSplit
Métrica de seleção: F1 médio
Comparação principal: modelo tabular vencedor do NB11
```

A escolha do `temporal_core` é deliberada: trata-se do conjunto de features associado ao vencedor do NB11 e, portanto, é o ponto de comparação mais forte e metodologicamente mais relevante para avaliar se a LSTM deve ou não deslocar o modelo tabular.

## Hiperparâmetros avaliados

O grid compacto avalia:

```text
sequence_length: 12, 24, 36
units: 8, 16, 32
dropout: 0.10, 0.20, 0.30
learning_rate: 0.001, 0.0005
```

Esse grid gera 54 configurações. Para tornar a execução viável, o NB13a adota um desenho em duas fases:

1. **Triagem inicial**: avalia todas as 54 configurações com uma semente fixa, usando TimeSeriesSplit.
2. **Confirmação**: reavalia as melhores configurações com três sementes, reduzindo a chance de conclusão baseada em uma única inicialização aleatória.

## Critério de decisão

A LSTM ajustada só será recomendada como candidata a substituir os escores do NB11 no NB14 se superar o modelo tabular vencedor do NB11 por margem mínima previamente definida:

```text
ganho mínimo requerido: +0,0300 em F1 médio TSCV
```

Caso contrário, o NB13a será interpretado como evidência de robustez da decisão anterior: mesmo após tuning compacto, o modelo tabular permanece a escolha principal para o fluxo decisório.

## Controle de vazamento de informação

O NB13a preserva a disciplina de causalidade do pipeline:

- usa `11_sequence_input_features.parquet` como entrada sequencial;
- usa `11_feature_sets.json` como whitelist de features;
- usa apenas features do conjunto `temporal_core`;
- não usa `scenario_label`, `bucket_id`, `eligible_supervised`, `y_nb11`, estados, limiares, scores ou predições como variáveis de entrada;
- ajusta o escalonamento somente no trecho de treino de cada fold;
- usa a coluna `eligible_supervised` apenas para selecionar endpoints válidos das sequências.

## Artefatos gerados

O NB13a grava apenas arquivos com prefixo `13a_`, incluindo:

```text
13a_lstm_tuning_stage1_results.csv
13a_lstm_tuning_stage1_agg.csv
13a_lstm_tuning_stage2_results.csv
13a_lstm_tuning_stage2_agg.csv
13a_lstm_tuning_fixed_results.csv
13a_lstm_tuning_fixed_results_agg.csv
13a_lstm_tuning_scores.parquet
13a_lstm_tuning_best_config.json
13a_lstm_tuning_summary.json
13a_lstm_tuning_conclusion.txt
```

Nenhum artefato do NB13 ou de notebooks anteriores é sobrescrito.


In [1]:

# ============================================================
# NB13a — Tuning Compacto da LSTM
# ------------------------------------------------------------
# Notebook intermediário derivado do NB13.
#
# Regras:
#   - Pode ler artefatos anteriores, inclusive do NB13.
#   - Não sobrescreve artefatos do notebook pai.
#   - Todos os outputs usam prefixo 13a_.
#
# Objetivo:
#   Avaliar se um tuning compacto de hiperparâmetros da LSTM
#   altera a decisão metodológica do NB13/NB11 para o NB14.
# ============================================================

from __future__ import annotations

import os
import re
import json
import math
import time
import shutil
import random
import warnings
from pathlib import Path
from datetime import datetime
from itertools import product
from typing import Dict, List, Tuple, Any, Optional

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Ambiente e bibliotecas
# ------------------------------------------------------------

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[AVISO] Não foi possível montar o Google Drive automaticamente: {exc}")

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

print("TensorFlow:", tf.__version__)

# ------------------------------------------------------------
# 2. Configuração geral
# ------------------------------------------------------------

RUN_MODE = "NB13A_COMPACT_TUNING"

# Diretórios principais
DEFAULT_REPORTS_PATH = Path("/content/drive/MyDrive/Mestrado/04-reports")
LOCAL_FALLBACK_REPORTS_PATH = Path.cwd()

REPORTS_PATH = DEFAULT_REPORTS_PATH if DEFAULT_REPORTS_PATH.exists() else LOCAL_FALLBACK_REPORTS_PATH
FIGURES_PATH = REPORTS_PATH / "figures_nb13a_lstm_tuning"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

# Entradas principais
SEQUENCE_INPUT_PATH = REPORTS_PATH / "11_sequence_input_features.parquet"
MODEL_INPUT_PATH = REPORTS_PATH / "11_model_input_features.parquet"
FEATURE_SETS_PATH = REPORTS_PATH / "11_feature_sets.json"
SCHEMA_PATH = REPORTS_PATH / "11_model_input_features_schema.json"
NB11_SUMMARY_PATH = REPORTS_PATH / "11_nb11_summary.json"
WINNER_MODEL_PATH = REPORTS_PATH / "11_winner_model.json"
NB13_SUMMARY_PATH = REPORTS_PATH / "13_lstm_summary.json"

# Saídas NB13a — nunca sobrescrever 13_*
OUT_STAGE1_RESULTS = REPORTS_PATH / "13a_lstm_tuning_stage1_results.csv"
OUT_STAGE1_AGG = REPORTS_PATH / "13a_lstm_tuning_stage1_agg.csv"
OUT_STAGE2_RESULTS = REPORTS_PATH / "13a_lstm_tuning_stage2_results.csv"
OUT_STAGE2_AGG = REPORTS_PATH / "13a_lstm_tuning_stage2_agg.csv"
OUT_FIXED_RESULTS = REPORTS_PATH / "13a_lstm_tuning_fixed_results.csv"
OUT_FIXED_AGG = REPORTS_PATH / "13a_lstm_tuning_fixed_results_agg.csv"
OUT_SCORES = REPORTS_PATH / "13a_lstm_tuning_scores.parquet"
OUT_BEST_CONFIG = REPORTS_PATH / "13a_lstm_tuning_best_config.json"
OUT_SUMMARY = REPORTS_PATH / "13a_lstm_tuning_summary.json"
OUT_CONCLUSION = REPORTS_PATH / "13a_lstm_tuning_conclusion.txt"

# Escopo metodológico
MAIN_SCENARIO_DEFAULT = "W5_K24_H12_P1_TRAIN_M2S"
FEATURE_SET_TO_TUNE = "temporal_core"

# Grid solicitado
SEQUENCE_LENGTH_GRID = [12, 24, 36]
LSTM_UNITS_GRID = [8, 16, 32]
DROPOUT_GRID = [0.10, 0.20, 0.30]
LEARNING_RATE_GRID = [0.001, 0.0005]

# Desenho em duas fases
STAGE1_SEEDS = [42]
STAGE2_SEEDS = [42, 123, 2026]
TOP_K_FOR_CONFIRMATION = 3

# Validação
TSCV_SPLITS = 5
FIXED_TRAIN_FRACTION = 0.80
VALIDATION_FRACTION_FROM_TRAIN = 0.20

# Treinamento
MAX_EPOCHS = 50
BATCH_SIZE = 128
EARLY_STOPPING_PATIENCE = 8
REDUCE_LR_PATIENCE = 4
CLASSIFICATION_THRESHOLD = 0.50

# Decisão
MIN_DELTA_F1_TO_FEED_NB14 = 0.0300
MIN_DELTA_F1_TO_CONSIDER_TUNING_GAIN = 0.0100

# Segurança operacional
BACKUP_PREVIOUS_13A_OUTPUTS = True
SAVE_KERAS_MODELS = False
KERAS_VERBOSE = 0

# ------------------------------------------------------------
# 3. Utilitários
# ------------------------------------------------------------

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (Path,)):
        return str(obj)
    if pd.isna(obj):
        return None
    return str(obj)

def load_json(path: Path, required: bool = True) -> Dict[str, Any]:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Arquivo não encontrado: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj: Dict[str, Any], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_default)

def backup_existing_13a_outputs() -> Optional[Path]:
    if not BACKUP_PREVIOUS_13A_OUTPUTS:
        return None

    patterns = [
        "13a_lstm_tuning_stage1_results.csv",
        "13a_lstm_tuning_stage1_agg.csv",
        "13a_lstm_tuning_stage2_results.csv",
        "13a_lstm_tuning_stage2_agg.csv",
        "13a_lstm_tuning_fixed_results.csv",
        "13a_lstm_tuning_fixed_results_agg.csv",
        "13a_lstm_tuning_scores.parquet",
        "13a_lstm_tuning_best_config.json",
        "13a_lstm_tuning_summary.json",
        "13a_lstm_tuning_conclusion.txt",
    ]

    existing = [REPORTS_PATH / p for p in patterns if (REPORTS_PATH / p).exists()]
    if not existing:
        return None

    backup_dir = REPORTS_PATH / "_backup_previous_artifacts" / f"nb13a_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    backup_dir.mkdir(parents=True, exist_ok=True)

    for p in existing:
        shutil.copy2(p, backup_dir / p.name)

    return backup_dir

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def safe_float(x) -> Optional[float]:
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None

def normalize_feature_sets(feature_sets_obj: Dict[str, Any], scenario_label: str, feature_set: str) -> List[str]:
    """
    Tenta extrair lista de features em diferentes estruturas possíveis do 11_feature_sets.json.
    Fallback explícito para temporal_core, se necessário.
    """
    # Estrutura 1: {"scenario": {"feature_set": [...]}}
    if scenario_label in feature_sets_obj:
        node = feature_sets_obj[scenario_label]
        if isinstance(node, dict) and feature_set in node and isinstance(node[feature_set], list):
            return list(node[feature_set])

    # Estrutura 2: {"feature_sets": {"scenario": {"feature_set": [...]}}}
    if "feature_sets" in feature_sets_obj:
        fs = feature_sets_obj["feature_sets"]
        if isinstance(fs, dict):
            if scenario_label in fs and isinstance(fs[scenario_label], dict):
                if feature_set in fs[scenario_label] and isinstance(fs[scenario_label][feature_set], list):
                    return list(fs[scenario_label][feature_set])
            if feature_set in fs and isinstance(fs[feature_set], list):
                return list(fs[feature_set])

    # Estrutura 3: {"raw_7": [...], "temporal_core": [...]}
    if feature_set in feature_sets_obj and isinstance(feature_sets_obj[feature_set], list):
        return list(feature_sets_obj[feature_set])

    # Estrutura 4: lista de planos
    for key in ["feature_plan", "plans", "scenarios"]:
        if key in feature_sets_obj and isinstance(feature_sets_obj[key], list):
            for item in feature_sets_obj[key]:
                if not isinstance(item, dict):
                    continue
                if item.get("scenario_label") == scenario_label and item.get("feature_set") == feature_set:
                    feats = item.get("features")
                    if isinstance(feats, list):
                        return feats

    # Fallback deliberado para temporal_core
    if feature_set == "temporal_core":
        return [
            "fail_rate",
            "n_events",
            "n_failed",
            "n_machines",
            "n_collections",
            "event_FAIL_count",
            "event_LOST_count",
            "lag_1",
            "lag_2",
            "lag_3",
            "rolling_mean_1h",
            "rolling_std_1h",
            "pct_change",
            "zscore_expanding",
        ]

    if feature_set == "raw_7":
        return [
            "fail_rate",
            "n_events",
            "n_failed",
            "n_machines",
            "n_collections",
            "event_FAIL_count",
            "event_LOST_count",
        ]

    raise ValueError(f"Não foi possível extrair features para feature_set={feature_set}")

def detect_schema_columns(schema: Dict[str, Any], df_columns: List[str]) -> Dict[str, str]:
    """
    Identifica colunas estruturais com base no schema e em fallbacks.
    """
    def pick(candidates: List[str], default: Optional[str] = None) -> str:
        for c in candidates:
            if c in df_columns:
                return c
        if default is not None:
            return default
        raise ValueError(f"Nenhuma coluna encontrada entre: {candidates}")

    # Tenta ler nomes declarados no schema, se existirem
    target_candidates = [
        schema.get("target_column"),
        schema.get("target_col"),
        "y_nb11",
        "y",
        "target",
    ]
    eligible_candidates = [
        schema.get("endpoint_eligibility_column"),
        schema.get("eligibility_column"),
        "eligible_supervised",
    ]
    scenario_candidates = [
        schema.get("scenario_column"),
        "scenario_label",
        "scenario",
    ]
    order_candidates = [
        schema.get("time_order_column"),
        schema.get("order_column"),
        "bucket_id",
        "row_position",
    ]

    target_candidates = [c for c in target_candidates if isinstance(c, str)]
    eligible_candidates = [c for c in eligible_candidates if isinstance(c, str)]
    scenario_candidates = [c for c in scenario_candidates if isinstance(c, str)]
    order_candidates = [c for c in order_candidates if isinstance(c, str)]

    return {
        "target": pick(target_candidates),
        "eligible": pick(eligible_candidates),
        "scenario": pick(scenario_candidates),
        "order": pick(order_candidates),
    }

def sanitize_features(features: List[str], df_columns: List[str], structural_cols: Dict[str, str]) -> List[str]:
    forbidden_exact = {
        structural_cols["target"],
        structural_cols["eligible"],
        structural_cols["scenario"],
        structural_cols["order"],
        "bucket_start",
        "bucket_start_datetime",
        "bucket_start_dt",
        "bucket_start_us",
        "timestamp",
        "datetime",
        "time",
        "state",
        "episode_state",
        "operational_state",
        "is_critical",
        "is_during",
        "is_before",
        "is_after",
        "is_normal",
        "threshold_value",
        "scenario_id",
        "scenario_key",
        "scenario_name",
        "y_nb13",
        "target_col",
    }

    forbidden_patterns = [
        r"^y$",
        r"^y_",
        r"target",
        r"label",
        r"future",
        r"will_",
        r"ahead",
        r"state",
        r"estado",
        r"phase",
        r"during",
        r"before",
        r"after",
        r"episode",
        r"critical",
        r"normal",
        r"eligible",
        r"horizon",
        r"persistence",
        r"threshold",
        r"limiar",
        r"scenario",
        r"cenario",
        r"bucket_start",
        r"timestamp",
        r"datetime",
        r"row_position",
        r"(^|_)score($|_)",
        r"(^|_)scores($|_)",
        r"(^|_)prob($|_)",
        r"(^|_)proba($|_)",
        r"(^|_)pred($|_)",
        r"(^|_)prediction($|_)",
        r"(^|_)fold($|_)",
        r"(^|_)split($|_)",
    ]

    clean = []
    missing = []
    rejected = []
    for feat in features:
        if feat not in df_columns:
            missing.append(feat)
            continue
        if feat in forbidden_exact:
            rejected.append((feat, "forbidden_exact"))
            continue
        if any(re.search(p, feat, flags=re.IGNORECASE) for p in forbidden_patterns):
            rejected.append((feat, "forbidden_pattern"))
            continue
        clean.append(feat)

    if missing:
        print("[AVISO] Features ausentes no dataframe:", missing)
    if rejected:
        print("[AVISO] Features rejeitadas por controle de leakage:", rejected)

    if not clean:
        raise ValueError("Nenhuma feature válida após controle de leakage.")

    return clean

def make_sequences_for_scenario(
    df: pd.DataFrame,
    features: List[str],
    target_col: str,
    eligible_col: str,
    order_col: str,
    sequence_length: int,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Forma sequências LSTM respeitando a ordem temporal.
    A elegibilidade é aplicada ao endpoint da sequência, não aos pontos internos.
    """
    sdf = df.sort_values(order_col).reset_index(drop=True).copy()

    X_raw = sdf[features].replace([np.inf, -np.inf], np.nan)
    X_raw = X_raw.ffill().fillna(0.0).astype("float32")
    y_raw = sdf[target_col].astype(int).values
    eligible = sdf[eligible_col].astype(bool).values

    X_seq = []
    y_seq = []
    meta_rows = []

    for end_idx in range(sequence_length - 1, len(sdf)):
        if not eligible[end_idx]:
            continue

        start_idx = end_idx - sequence_length + 1
        seq = X_raw.iloc[start_idx:end_idx + 1].values

        X_seq.append(seq)
        y_seq.append(y_raw[end_idx])
        meta_rows.append({
            "row_position": int(end_idx),
            "bucket_id": int(sdf.loc[end_idx, order_col]) if pd.notna(sdf.loc[end_idx, order_col]) else int(end_idx),
            "y_true": int(y_raw[end_idx]),
        })

    X_seq = np.asarray(X_seq, dtype="float32")
    y_seq = np.asarray(y_seq, dtype=int)
    meta = pd.DataFrame(meta_rows)

    return X_seq, y_seq, meta

def temporal_train_val_split(X_train: np.ndarray, y_train: np.ndarray, val_fraction: float):
    n = len(y_train)
    n_val = max(1, int(round(n * val_fraction)))
    n_train_inner = n - n_val

    # Garante pelo menos algumas amostras para treino
    if n_train_inner < 50:
        n_train_inner = max(1, n - 1)
        n_val = n - n_train_inner

    return (
        X_train[:n_train_inner],
        y_train[:n_train_inner],
        X_train[n_train_inner:],
        y_train[n_train_inner:],
    )

def scale_sequences(X_train: np.ndarray, X_val: np.ndarray, X_test: np.ndarray):
    n_train, seq_len, n_feat = X_train.shape

    scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, n_feat)
    scaler.fit(X_train_2d)

    def transform(X):
        shape = X.shape
        return scaler.transform(X.reshape(-1, n_feat)).reshape(shape).astype("float32")

    return transform(X_train), transform(X_val), transform(X_test), scaler

def compute_class_weight_dict(y: np.ndarray) -> Optional[Dict[int, float]]:
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2:
        return None
    n = len(y)
    weights = {int(cls): float(n / (len(classes) * cnt)) for cls, cnt in zip(classes, counts)}
    return weights

def build_lstm_model(sequence_length: int, n_features: int, units: int, dropout: float, learning_rate: float):
    model = models.Sequential([
        layers.Input(shape=(sequence_length, n_features)),
        layers.LSTM(units),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[],
    )
    return model

def calculate_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> Dict[str, Any]:
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "n": int(len(y_true)),
        "n_positive": int(np.sum(y_true)),
        "positive_rate": float(np.mean(y_true)) if len(y_true) else None,
        "accuracy": safe_float(accuracy_score(y_true, y_pred)),
        "precision": safe_float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": safe_float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": safe_float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": safe_float(brier_score_loss(y_true, y_score)),
    }

    try:
        out["roc_auc"] = safe_float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else None
    except Exception:
        out["roc_auc"] = None

    try:
        out["average_precision"] = safe_float(average_precision_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else None
    except Exception:
        out["average_precision"] = None

    return out

def train_and_evaluate_once(
    X_train_full: np.ndarray,
    y_train_full: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    sequence_length: int,
    n_features: int,
    units: int,
    dropout: float,
    learning_rate: float,
    seed: int,
) -> Dict[str, Any]:
    set_global_seed(seed)
    tf.keras.backend.clear_session()

    X_train_inner, y_train_inner, X_val, y_val = temporal_train_val_split(
        X_train_full, y_train_full, VALIDATION_FRACTION_FROM_TRAIN
    )

    X_train_scaled, X_val_scaled, X_test_scaled, _ = scale_sequences(X_train_inner, X_val, X_test)

    model = build_lstm_model(sequence_length, n_features, units, dropout, learning_rate)

    cb = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            min_delta=1e-5,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            patience=REDUCE_LR_PATIENCE,
            factor=0.5,
            min_lr=1e-6,
        ),
    ]

    class_weight = compute_class_weight_dict(y_train_inner)

    t0 = time.time()
    hist = model.fit(
        X_train_scaled,
        y_train_inner,
        validation_data=(X_val_scaled, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=KERAS_VERBOSE,
        callbacks=cb,
        class_weight=class_weight,
        shuffle=False,
    )
    fit_elapsed = time.time() - t0

    y_score = model.predict(X_test_scaled, verbose=0).reshape(-1)
    metrics = calculate_metrics(y_test, y_score, CLASSIFICATION_THRESHOLD)

    metrics.update({
        "epochs_ran": int(len(hist.history.get("loss", []))),
        "best_val_loss": float(np.min(hist.history.get("val_loss", [np.nan]))),
        "fit_elapsed_seconds": float(fit_elapsed),
        "total_elapsed_seconds": float(time.time() - t0),
    })

    return metrics | {"y_score": y_score}

def aggregate_results(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    metric_cols = [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
        "brier",
        "n",
        "n_positive",
        "positive_rate",
        "epochs_ran",
        "best_val_loss",
        "fit_elapsed_seconds",
        "total_elapsed_seconds",
    ]

    agg_map = {}
    for c in metric_cols:
        if c in df.columns:
            agg_map[c] = ["mean", "std"]

    out = df.groupby(group_cols, dropna=False).agg(agg_map)
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out = out.reset_index()

    # Ordenação para seleção
    sort_cols = []
    for c in ["f1_mean", "recall_mean", "roc_auc_mean", "average_precision_mean"]:
        if c in out.columns:
            sort_cols.append(c)

    if sort_cols:
        out = out.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

    return out

def run_tscv_for_config(
    X: np.ndarray,
    y: np.ndarray,
    config: Dict[str, Any],
    seeds: List[int],
    stage: str,
) -> pd.DataFrame:
    rows = []
    tscv = TimeSeriesSplit(n_splits=TSCV_SPLITS)

    for seed in seeds:
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
            y_train = y[train_idx]
            y_test = y[test_idx]

            if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                row = {
                    "stage": stage,
                    "seed": seed,
                    "fold": fold,
                    "valid_fold": False,
                    "skip_reason": "single_class_train_or_test",
                    **config,
                }
                rows.append(row)
                continue

            result = train_and_evaluate_once(
                X_train_full=X[train_idx],
                y_train_full=y_train,
                X_test=X[test_idx],
                y_test=y_test,
                sequence_length=config["sequence_length"],
                n_features=X.shape[-1],
                units=config["units"],
                dropout=config["dropout"],
                learning_rate=config["learning_rate"],
                seed=seed,
            )

            y_score = result.pop("y_score")
            row = {
                "stage": stage,
                "seed": seed,
                "fold": fold,
                "valid_fold": True,
                "skip_reason": "",
                **config,
                **result,
            }
            rows.append(row)

    return pd.DataFrame(rows)

def run_fixed_for_config(
    X: np.ndarray,
    y: np.ndarray,
    meta: pd.DataFrame,
    config: Dict[str, Any],
    seeds: List[int],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n = len(y)
    split_idx = int(math.floor(n * FIXED_TRAIN_FRACTION))

    X_train = X[:split_idx]
    y_train = y[:split_idx]
    X_test = X[split_idx:]
    y_test = y[split_idx:]
    meta_test = meta.iloc[split_idx:].reset_index(drop=True).copy()

    rows = []
    score_frames = []

    for seed in seeds:
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            rows.append({
                "seed": seed,
                "valid_split": False,
                "skip_reason": "single_class_train_or_test",
                **config,
            })
            continue

        result = train_and_evaluate_once(
            X_train_full=X_train,
            y_train_full=y_train,
            X_test=X_test,
            y_test=y_test,
            sequence_length=config["sequence_length"],
            n_features=X.shape[-1],
            units=config["units"],
            dropout=config["dropout"],
            learning_rate=config["learning_rate"],
            seed=seed,
        )

        y_score = result.pop("y_score")
        rows.append({
            "seed": seed,
            "valid_split": True,
            "skip_reason": "",
            **config,
            **result,
        })

        sf = meta_test.copy()
        sf["seed"] = seed
        sf["score_lstm_tuned"] = y_score
        sf["y_pred_lstm_tuned"] = (y_score >= CLASSIFICATION_THRESHOLD).astype(int)
        for k, v in config.items():
            sf[k] = v
        score_frames.append(sf)

    fixed_results = pd.DataFrame(rows)
    scores = pd.concat(score_frames, ignore_index=True) if score_frames else pd.DataFrame()
    return fixed_results, scores

# ------------------------------------------------------------
# 4. Backup dos outputs 13a anteriores
# ------------------------------------------------------------

backup_dir = backup_existing_13a_outputs()
if backup_dir:
    print(f"[INFO] Outputs 13a anteriores copiados para: {backup_dir}")

# ------------------------------------------------------------
# 5. Leitura dos artefatos de entrada
# ------------------------------------------------------------

required_inputs = [
    SEQUENCE_INPUT_PATH,
    FEATURE_SETS_PATH,
]

for p in required_inputs:
    if not p.exists():
        raise FileNotFoundError(f"Entrada obrigatória não encontrada: {p}")

print("[INFO] Lendo artefatos...")
df_seq = pd.read_parquet(SEQUENCE_INPUT_PATH)
feature_sets_obj = load_json(FEATURE_SETS_PATH)
schema_obj = load_json(SCHEMA_PATH, required=False)
nb11_summary = load_json(NB11_SUMMARY_PATH, required=False)
winner_model = load_json(WINNER_MODEL_PATH, required=False)
nb13_summary = load_json(NB13_SUMMARY_PATH, required=False)

struct_cols = detect_schema_columns(schema_obj, list(df_seq.columns))
print("[INFO] Colunas estruturais:", struct_cols)

main_scenario = (
    winner_model.get("scenario_label")
    or winner_model.get("scenario")
    or nb11_summary.get("winner", {}).get("scenario_label")
    or MAIN_SCENARIO_DEFAULT
)

print("[INFO] Cenário principal:", main_scenario)

raw_features = normalize_feature_sets(feature_sets_obj, main_scenario, FEATURE_SET_TO_TUNE)
features = sanitize_features(raw_features, list(df_seq.columns), struct_cols)

print(f"[INFO] Feature set avaliado: {FEATURE_SET_TO_TUNE} ({len(features)} features)")
print(features)

# ------------------------------------------------------------
# 6. Preparação da base sequencial
# ------------------------------------------------------------

scenario_col = struct_cols["scenario"]
target_col = struct_cols["target"]
eligible_col = struct_cols["eligible"]
order_col = struct_cols["order"]

df_main = df_seq[df_seq[scenario_col] == main_scenario].copy()
if df_main.empty:
    raise ValueError(f"Nenhuma linha encontrada para scenario_label={main_scenario}")

df_main = df_main.sort_values(order_col).reset_index(drop=True)

diagnostics_by_L = {}
sequence_data = {}

for L in SEQUENCE_LENGTH_GRID:
    X, y, meta = make_sequences_for_scenario(
        df=df_main,
        features=features,
        target_col=target_col,
        eligible_col=eligible_col,
        order_col=order_col,
        sequence_length=L,
    )
    if len(y) == 0:
        raise ValueError(f"Nenhuma sequência formada para sequence_length={L}")

    diagnostics_by_L[str(L)] = {
        "sequence_length": L,
        "n_sequences": int(len(y)),
        "n_positive": int(np.sum(y)),
        "positive_rate": float(np.mean(y)),
        "n_features": int(X.shape[-1]),
    }
    sequence_data[L] = (X, y, meta)

print("[INFO] Diagnóstico por sequence_length:")
print(json.dumps(diagnostics_by_L, ensure_ascii=False, indent=2))

# ------------------------------------------------------------
# 7. Referências NB11 e NB13
# ------------------------------------------------------------

def extract_nb11_reference(nb11_summary: Dict[str, Any], winner_model: Dict[str, Any]) -> Dict[str, Any]:
    # Preferir winner_model
    candidates = [winner_model, nb11_summary.get("winner_model", {}), nb11_summary.get("winner", {})]
    ref = {}
    for cand in candidates:
        if not isinstance(cand, dict):
            continue
        if cand:
            ref = cand
            break

    f1 = (
        ref.get("f1_tscv_mean")
        or ref.get("f1_mean")
        or ref.get("tscv", {}).get("f1_mean")
        or ref.get("metrics", {}).get("f1_tscv_mean")
        or nb11_summary.get("best_model", {}).get("f1_tscv_mean")
    )

    # fallback conhecido pela execução atual do NB11
    if f1 is None:
        f1 = 0.5773731424098736

    return {
        "scenario_label": ref.get("scenario_label", main_scenario),
        "model": ref.get("model", ref.get("model_name", "logistic_regression")),
        "feature_set": ref.get("feature_set", FEATURE_SET_TO_TUNE),
        "f1_tscv_mean": float(f1),
    }

def extract_nb13_reference(nb13_summary: Dict[str, Any]) -> Dict[str, Any]:
    best = nb13_summary.get("best_lstm_tscv", {})
    f1 = nb13_summary.get("best_lstm_f1_tscv_mean") or best.get("f1_mean")
    if f1 is None:
        # fallback da execução NB13 atual
        f1 = 0.5579689978151948
    return {
        "available": bool(nb13_summary),
        "best_lstm_tscv": best,
        "best_lstm_f1_tscv_mean": float(f1),
    }

nb11_reference = extract_nb11_reference(nb11_summary, winner_model)
nb13_reference = extract_nb13_reference(nb13_summary)

print("[INFO] Referência NB11:", nb11_reference)
print("[INFO] Referência NB13:", nb13_reference)

# ------------------------------------------------------------
# 8. Stage 1 — triagem do grid completo
# ------------------------------------------------------------

grid_configs = [
    {
        "scenario_label": main_scenario,
        "feature_set": FEATURE_SET_TO_TUNE,
        "sequence_length": L,
        "units": u,
        "dropout": d,
        "learning_rate": lr,
    }
    for L, u, d, lr in product(SEQUENCE_LENGTH_GRID, LSTM_UNITS_GRID, DROPOUT_GRID, LEARNING_RATE_GRID)
]

print(f"[INFO] Stage 1: {len(grid_configs)} configurações × {len(STAGE1_SEEDS)} seed(s) × {TSCV_SPLITS} folds")

stage1_frames = []
stage1_start = time.time()

for idx, cfg in enumerate(grid_configs, start=1):
    print(f"[STAGE1] {idx:03d}/{len(grid_configs)} — L={cfg['sequence_length']} units={cfg['units']} dropout={cfg['dropout']} lr={cfg['learning_rate']}")
    X, y, _meta = sequence_data[cfg["sequence_length"]]
    df_res = run_tscv_for_config(X, y, cfg, STAGE1_SEEDS, stage="stage1_screening")
    stage1_frames.append(df_res)

stage1_results = pd.concat(stage1_frames, ignore_index=True)
stage1_results.to_csv(OUT_STAGE1_RESULTS, index=False)

valid_stage1 = stage1_results[stage1_results["valid_fold"] == True].copy()
stage1_agg = aggregate_results(
    valid_stage1,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
stage1_agg.to_csv(OUT_STAGE1_AGG, index=False)

print(f"[INFO] Stage 1 concluído em {(time.time() - stage1_start)/60:.2f} min")
print("[INFO] Top configurações Stage 1:")
print(stage1_agg.head(TOP_K_FOR_CONFIRMATION)[[
    "sequence_length", "units", "dropout", "learning_rate", "f1_mean", "recall_mean", "precision_mean", "roc_auc_mean"
]])

# ------------------------------------------------------------
# 9. Stage 2 — confirmação das melhores configurações com múltiplas sementes
# ------------------------------------------------------------

top_configs_records = stage1_agg.head(TOP_K_FOR_CONFIRMATION)[
    ["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"]
].to_dict(orient="records")

print(f"[INFO] Stage 2: top {len(top_configs_records)} configurações × {len(STAGE2_SEEDS)} seeds × {TSCV_SPLITS} folds")

stage2_frames = []
stage2_start = time.time()

for idx, cfg in enumerate(top_configs_records, start=1):
    print(f"[STAGE2] {idx:03d}/{len(top_configs_records)} — L={cfg['sequence_length']} units={cfg['units']} dropout={cfg['dropout']} lr={cfg['learning_rate']}")
    X, y, _meta = sequence_data[int(cfg["sequence_length"])]
    df_res = run_tscv_for_config(X, y, cfg, STAGE2_SEEDS, stage="stage2_confirmation")
    stage2_frames.append(df_res)

stage2_results = pd.concat(stage2_frames, ignore_index=True)
stage2_results.to_csv(OUT_STAGE2_RESULTS, index=False)

valid_stage2 = stage2_results[stage2_results["valid_fold"] == True].copy()
stage2_agg = aggregate_results(
    valid_stage2,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
stage2_agg.to_csv(OUT_STAGE2_AGG, index=False)

print(f"[INFO] Stage 2 concluído em {(time.time() - stage2_start)/60:.2f} min")
print("[INFO] Top configurações Stage 2:")
print(stage2_agg.head(10)[[
    "sequence_length", "units", "dropout", "learning_rate", "f1_mean", "f1_std", "recall_mean", "precision_mean", "roc_auc_mean"
]])

# ------------------------------------------------------------
# 10. Seleção final e avaliação complementar no corte fixo
# ------------------------------------------------------------

if stage2_agg.empty:
    raise RuntimeError("Stage 2 não produziu resultados válidos.")

best_row = stage2_agg.iloc[0].to_dict()

best_config = {
    "scenario_label": main_scenario,
    "feature_set": FEATURE_SET_TO_TUNE,
    "sequence_length": int(best_row["sequence_length"]),
    "units": int(best_row["units"]),
    "dropout": float(best_row["dropout"]),
    "learning_rate": float(best_row["learning_rate"]),
}

X_best, y_best, meta_best = sequence_data[best_config["sequence_length"]]
fixed_results, scores = run_fixed_for_config(X_best, y_best, meta_best, best_config, STAGE2_SEEDS)
fixed_results.to_csv(OUT_FIXED_RESULTS, index=False)

valid_fixed = fixed_results[fixed_results["valid_split"] == True].copy()
fixed_agg = aggregate_results(
    valid_fixed,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
fixed_agg.to_csv(OUT_FIXED_AGG, index=False)

if not scores.empty:
    # Cria escore médio por endpoint para facilitar análise visual futura
    score_mean = (
        scores.groupby(["row_position", "bucket_id", "y_true"], as_index=False)
        .agg(
            score_lstm_tuned_mean=("score_lstm_tuned", "mean"),
            score_lstm_tuned_std=("score_lstm_tuned", "std"),
        )
    )
    for k, v in best_config.items():
        score_mean[k] = v
    score_mean["classification_threshold"] = CLASSIFICATION_THRESHOLD
    score_mean["y_pred_lstm_tuned_mean"] = (score_mean["score_lstm_tuned_mean"] >= CLASSIFICATION_THRESHOLD).astype(int)
    score_mean.to_parquet(OUT_SCORES, index=False)

# ------------------------------------------------------------
# 11. Decisão e síntese
# ------------------------------------------------------------

best_stage2_metrics = {
    k: safe_float(best_row.get(k))
    for k in [
        "accuracy_mean", "accuracy_std",
        "precision_mean", "precision_std",
        "recall_mean", "recall_std",
        "f1_mean", "f1_std",
        "roc_auc_mean", "roc_auc_std",
        "average_precision_mean", "average_precision_std",
        "brier_mean", "brier_std",
        "epochs_ran_mean", "epochs_ran_std",
        "best_val_loss_mean", "best_val_loss_std",
        "fit_elapsed_seconds_mean", "total_elapsed_seconds_mean",
    ]
}

best_f1 = best_stage2_metrics.get("f1_mean")
nb11_f1 = nb11_reference["f1_tscv_mean"]
nb13_f1 = nb13_reference["best_lstm_f1_tscv_mean"]

delta_vs_nb11 = best_f1 - nb11_f1 if best_f1 is not None else None
delta_vs_nb13 = best_f1 - nb13_f1 if best_f1 is not None else None

if delta_vs_nb11 is not None and delta_vs_nb11 >= MIN_DELTA_F1_TO_FEED_NB14:
    feed_nb14_decision = "candidate_lstm_tuned_scores_for_nb14"
else:
    feed_nb14_decision = "keep_nb11_scores_for_nb14"

if delta_vs_nb13 is not None and delta_vs_nb13 >= MIN_DELTA_F1_TO_CONSIDER_TUNING_GAIN:
    tuning_interpretation = "compact_tuning_improved_lstm_materially"
elif delta_vs_nb13 is not None and delta_vs_nb13 > 0:
    tuning_interpretation = "compact_tuning_improved_lstm_slightly"
else:
    tuning_interpretation = "compact_tuning_did_not_improve_lstm"

best_config_payload = {
    "created_at": now_str(),
    "selection_protocol": "stage2_confirmation_timeseries_split",
    "selection_metric": "f1_mean",
    "best_config": best_config,
    "best_stage2_metrics": best_stage2_metrics,
    "nb11_reference": nb11_reference,
    "nb13_reference": nb13_reference,
    "delta_f1_tuned_lstm_minus_nb11": delta_vs_nb11,
    "delta_f1_tuned_lstm_minus_nb13": delta_vs_nb13,
    "min_delta_f1_to_feed_nb14": MIN_DELTA_F1_TO_FEED_NB14,
    "feed_nb14_decision": feed_nb14_decision,
    "tuning_interpretation": tuning_interpretation,
}
save_json(best_config_payload, OUT_BEST_CONFIG)

summary = {
    "notebook": "NB13a",
    "title": "Tuning compacto da LSTM",
    "executed_at": now_str(),
    "objective": (
        "Avaliar se um tuning compacto de hiperparâmetros da LSTM altera a conclusão do NB13 "
        "sobre a manutenção dos escores do modelo tabular vencedor do NB11 no NB14."
    ),
    "intermediate_notebook_rule": {
        "can_read_previous_artifacts": True,
        "must_not_overwrite_parent_outputs": True,
        "output_prefix": "13a_",
    },
    "config": {
        "run_mode": RUN_MODE,
        "main_scenario": main_scenario,
        "feature_set": FEATURE_SET_TO_TUNE,
        "sequence_length_grid": SEQUENCE_LENGTH_GRID,
        "lstm_units_grid": LSTM_UNITS_GRID,
        "dropout_grid": DROPOUT_GRID,
        "learning_rate_grid": LEARNING_RATE_GRID,
        "stage1_seeds": STAGE1_SEEDS,
        "stage2_seeds": STAGE2_SEEDS,
        "top_k_for_confirmation": TOP_K_FOR_CONFIRMATION,
        "tscv_splits": TSCV_SPLITS,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "fixed_train_fraction": FIXED_TRAIN_FRACTION,
        "validation_fraction_from_train": VALIDATION_FRACTION_FROM_TRAIN,
        "min_delta_f1_to_feed_nb14": MIN_DELTA_F1_TO_FEED_NB14,
    },
    "input_artifacts": {
        "sequence_input_features": str(SEQUENCE_INPUT_PATH),
        "model_input_features": str(MODEL_INPUT_PATH),
        "feature_sets": str(FEATURE_SETS_PATH),
        "schema": str(SCHEMA_PATH),
        "nb11_summary": str(NB11_SUMMARY_PATH),
        "winner_model": str(WINNER_MODEL_PATH),
        "nb13_summary": str(NB13_SUMMARY_PATH),
    },
    "observed_shapes": {
        "sequence_input_shape": list(df_seq.shape),
        "main_scenario_rows": int(len(df_main)),
    },
    "structural_columns": struct_cols,
    "feature_plan": {
        "feature_set": FEATURE_SET_TO_TUNE,
        "n_features": len(features),
        "features": features,
    },
    "diagnostics_by_sequence_length": diagnostics_by_L,
    "estimated_trainings": {
        "stage1": int(len(grid_configs) * len(STAGE1_SEEDS) * TSCV_SPLITS),
        "stage2": int(len(top_configs_records) * len(STAGE2_SEEDS) * TSCV_SPLITS),
        "fixed_complementary": int(len(STAGE2_SEEDS)),
        "total_estimated": int(len(grid_configs) * len(STAGE1_SEEDS) * TSCV_SPLITS + len(top_configs_records) * len(STAGE2_SEEDS) * TSCV_SPLITS + len(STAGE2_SEEDS)),
    },
    "actual_results": {
        "stage1_rows": int(len(stage1_results)),
        "stage1_valid_rows": int(len(valid_stage1)),
        "stage2_rows": int(len(stage2_results)),
        "stage2_valid_rows": int(len(valid_stage2)),
        "fixed_rows": int(len(fixed_results)),
        "fixed_valid_rows": int(len(valid_fixed)),
    },
    "best_config": best_config,
    "best_stage2_metrics": best_stage2_metrics,
    "nb11_reference": nb11_reference,
    "nb13_reference": nb13_reference,
    "delta_f1_tuned_lstm_minus_nb11": delta_vs_nb11,
    "delta_f1_tuned_lstm_minus_nb13": delta_vs_nb13,
    "feed_nb14_decision": feed_nb14_decision,
    "tuning_interpretation": tuning_interpretation,
    "outputs": {
        "stage1_results": str(OUT_STAGE1_RESULTS),
        "stage1_agg": str(OUT_STAGE1_AGG),
        "stage2_results": str(OUT_STAGE2_RESULTS),
        "stage2_agg": str(OUT_STAGE2_AGG),
        "fixed_results": str(OUT_FIXED_RESULTS),
        "fixed_agg": str(OUT_FIXED_AGG),
        "scores": str(OUT_SCORES),
        "best_config": str(OUT_BEST_CONFIG),
        "summary": str(OUT_SUMMARY),
        "conclusion": str(OUT_CONCLUSION),
    },
    "backup_previous_13a_outputs": str(backup_dir) if backup_dir else None,
    "methodological_notes": [
        "O tuning é restrito ao cenário principal e ao feature set temporal_core, pois este é o ponto de comparação mais forte com o vencedor do NB11.",
        "A seleção usa TimeSeriesSplit como protocolo principal.",
        "O corte fixo 80/20 é complementar e aplicado apenas à configuração final selecionada.",
        "O escalonamento é ajustado apenas no trecho de treino de cada fold.",
        "A validação interna para early stopping usa a parte final do treino, preservando a ordem temporal.",
        "A coluna eligible_supervised é usada apenas para selecionar endpoints elegíveis, não como feature.",
        "Nenhum arquivo 13_ ou de notebooks anteriores é sobrescrito.",
    ],
}
save_json(summary, OUT_SUMMARY)

# Conclusão textual
def fmt(v, nd=4):
    if v is None:
        return "n/d"
    try:
        return f"{float(v):.{nd}f}"
    except Exception:
        return str(v)

conclusion_lines = [
    "Conclusão do NB13a:",
    "",
    "O NB13a executou um tuning compacto da LSTM como notebook intermediário derivado do NB13, sem sobrescrever artefatos do notebook pai.",
    "",
    f"O escopo foi restrito ao cenário principal {main_scenario} e ao feature set {FEATURE_SET_TO_TUNE}.",
    f"A triagem avaliou {len(grid_configs)} configurações com {len(STAGE1_SEEDS)} semente(s) e TimeSeriesSplit com {TSCV_SPLITS} folds.",
    f"A confirmação reavaliou as {TOP_K_FOR_CONFIRMATION} melhores configurações com {len(STAGE2_SEEDS)} sementes.",
    "",
    "Melhor configuração confirmada:",
    f"- sequence_length = {best_config['sequence_length']}",
    f"- units = {best_config['units']}",
    f"- dropout = {best_config['dropout']}",
    f"- learning_rate = {best_config['learning_rate']}",
    "",
    "Desempenho da melhor LSTM ajustada no TimeSeriesSplit:",
    f"- F1 médio = {fmt(best_stage2_metrics.get('f1_mean'))}",
    f"- desvio-padrão do F1 = {fmt(best_stage2_metrics.get('f1_std'))}",
    f"- recall médio = {fmt(best_stage2_metrics.get('recall_mean'))}",
    f"- precisão média = {fmt(best_stage2_metrics.get('precision_mean'))}",
    f"- ROC-AUC médio = {fmt(best_stage2_metrics.get('roc_auc_mean'))}",
    f"- Brier médio = {fmt(best_stage2_metrics.get('brier_mean'))}",
    "",
    f"Referência NB11: F1 médio TSCV = {fmt(nb11_f1)}.",
    f"Referência NB13: melhor F1 médio TSCV da LSTM anterior = {fmt(nb13_f1)}.",
    f"Delta da LSTM ajustada contra NB11 = {fmt(delta_vs_nb11)}.",
    f"Delta da LSTM ajustada contra NB13 = {fmt(delta_vs_nb13)}.",
    "",
    f"Decisão para NB14: {feed_nb14_decision}.",
    "",
]

if feed_nb14_decision == "candidate_lstm_tuned_scores_for_nb14":
    conclusion_lines.append(
        "A LSTM ajustada superou o modelo tabular vencedor do NB11 pela margem mínima definida. "
        "Recomenda-se avaliar, no NB14, se seus escores devem ser considerados como alternativa decisória."
    )
else:
    conclusion_lines.append(
        "A LSTM ajustada não superou o modelo tabular vencedor do NB11 pela margem mínima definida. "
        "Assim, os escores do NB11 permanecem como entrada preferencial do NB14."
    )

conclusion_lines.extend([
    "",
    "O resultado deve ser interpretado como tuning compacto e controlado, não como busca neural exaustiva.",
    "Ele fortalece a discussão metodológica da dissertação ao mostrar se a conclusão do NB13 se mantém ou não após ajuste moderado de hiperparâmetros.",
])

with open(OUT_CONCLUSION, "w", encoding="utf-8") as f:
    f.write("\n".join(conclusion_lines))

print("\n".join(conclusion_lines))

# ------------------------------------------------------------
# 12. Conferência final de outputs
# ------------------------------------------------------------

declared_outputs = [
    OUT_STAGE1_RESULTS,
    OUT_STAGE1_AGG,
    OUT_STAGE2_RESULTS,
    OUT_STAGE2_AGG,
    OUT_FIXED_RESULTS,
    OUT_FIXED_AGG,
    OUT_BEST_CONFIG,
    OUT_SUMMARY,
    OUT_CONCLUSION,
]

# scores pode não existir se fixed falhou; normalmente existirá
if OUT_SCORES.exists():
    declared_outputs.append(OUT_SCORES)

missing = [str(p) for p in declared_outputs if not p.exists()]
if missing:
    print("[AVISO] Outputs ausentes:", missing)
else:
    print("[OK] Todos os outputs declarados foram gerados.")


Mounted at /content/drive
TensorFlow: 2.20.0
[INFO] Lendo artefatos...
[INFO] Colunas estruturais: {'target': 'y_nb11', 'eligible': 'eligible_supervised', 'scenario': 'scenario_label', 'order': 'bucket_id'}
[INFO] Cenário principal: W5_K24_H12_P1_TRAIN_M2S
[INFO] Feature set avaliado: temporal_core (14 features)
['fail_rate', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'event_FAIL_count', 'event_LOST_count', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_1h', 'rolling_std_1h', 'pct_change', 'zscore_expanding']
[INFO] Diagnóstico por sequence_length:
{
  "12": {
    "sequence_length": 12,
    "n_sequences": 6954,
    "n_positive": 2605,
    "positive_rate": 0.37460454414725336,
    "n_features": 14
  },
  "24": {
    "sequence_length": 24,
    "n_sequences": 6954,
    "n_positive": 2605,
    "positive_rate": 0.37460454414725336,
    "n_features": 14
  },
  "36": {
    "sequence_length": 36,
    "n_sequences": 6944,
    "n_positive": 2605,
    "positive_rate": 0.37514400921658986,
 

## 8. Achados Experimentais da Execução do NB13a

### 8.1 Síntese da execução completa

O NB13a foi executado como notebook intermediário derivado do NB13, com o objetivo de avaliar se um tuning compacto de hiperparâmetros da LSTM alteraria a conclusão anterior de manter os escores do modelo tabular vencedor do NB11 como entrada preferencial do NB14.

A execução respeitou a regra metodológica definida para notebooks intermediários:

| Regra | Situação |
|---|---|
| Pode ler artefatos anteriores, inclusive do notebook pai | Cumprida |
| Não pode sobrescrever artefatos do notebook pai | Cumprida |
| Deve gerar artefatos com prefixo próprio | Cumprida — prefixo `13a_` |
| Deve preservar rastreabilidade | Cumprida por `summary`, `best_config`, resultados CSV e conclusão textual |

O NB13a manteve escopo controlado e não tentou transformar a dissertação em um estudo amplo de otimização neural. O tuning foi restrito ao cenário principal e ao conjunto de features mais diretamente comparável ao vencedor do NB11:

| Item | Valor |
|---|---|
| Notebook | NB13a |
| Nome simples | `13a_lstm_tuning_compacto` |
| Papel | Tuning compacto da LSTM |
| Cenário avaliado | `W5_K24_H12_P1_TRAIN_M2S` |
| Feature set | `temporal_core` |
| Protocolo principal | `TimeSeriesSplit` |
| Métrica de seleção | F1 médio |
| Referência NB11 | Regressão Logística / `temporal_core` |
| Referência NB13 | Melhor LSTM anterior |
| Decisão esperada | Confirmar ou ajustar `keep_nb11_scores_for_nb14` |

A execução foi concluída integralmente, com artefatos de triagem, confirmação, corte fixo complementar, melhor configuração, sumário consolidado, escores e conclusão textual.

## 9. Integridade dos Artefatos Gerados

Foram anexados os seguintes artefatos:

| Artefato | Finalidade | Situação |
|---|---|---|
| `13a_lstm_tuning_compacto.ipynb` | Notebook completo do NB13a | Gerado |
| `13a_lstm_tuning_stage1_results.csv` | Resultados detalhados da triagem por fold | Gerado |
| `13a_lstm_tuning_stage1_agg.csv` | Resultados agregados da triagem | Gerado |
| `13a_lstm_tuning_stage2_results.csv` | Resultados detalhados da confirmação por fold e seed | Gerado |
| `13a_lstm_tuning_stage2_agg.csv` | Resultados agregados da confirmação | Gerado |
| `13a_lstm_tuning_fixed_results.csv` | Resultados do corte fixo complementar | Gerado |
| `13a_lstm_tuning_fixed_results_agg.csv` | Consolidação do corte fixo | Gerado |
| `13a_lstm_tuning_scores.parquet` | Escores da configuração selecionada | Gerado |
| `13a_lstm_tuning_best_config.json` | Melhor configuração confirmada | Gerado |
| `13a_lstm_tuning_summary.json` | Sumário consolidado da execução | Gerado |
| `13a_lstm_tuning_conclusion.txt` | Conclusão textual automática | Gerado |

As quantidades registradas nos artefatos são coerentes com o desenho experimental:

| Etapa | Esperado | Realizado | Validação |
|---|---:|---:|---|
| Stage 1 — triagem | 270 avaliações | 270 válidas | OK |
| Stage 2 — confirmação | 45 avaliações | 45 válidas | OK |
| Corte fixo complementar | 3 avaliações | 3 válidas | OK |
| Total | 318 treinamentos/avaliações | 318 válidos | OK |

A execução, portanto, está completa e consistente com o plano do NB13a.

## 10. Entradas, Cenário e Feature Set

O NB13a utilizou os artefatos materializados pelo NB11 e pelo NB13 como entradas de rastreabilidade, sem sobrescrever os outputs anteriores.

As principais entradas foram:

```text
11_sequence_input_features.parquet
11_model_input_features.parquet
11_feature_sets.json
11_model_input_features_schema.json
11_nb11_summary.json
11_winner_model.json
13_lstm_summary.json
```

A entrada principal para construção das sequências foi:

```text
11_sequence_input_features.parquet
```

com dimensão observada:

| Artefato | Dimensão |
|---|---:|
| `11_sequence_input_features.parquet` | 62.510 × 46 |

O cenário principal avaliado foi:

```text
W5_K24_H12_P1_TRAIN_M2S
```

Esse cenário representa:

| Componente | Significado |
|---|---|
| `W5` | janela temporal de 5 minutos |
| `K24` | região BEFORE/AFTER de 24 janelas |
| `H12` | horizonte supervisionado de 12 janelas |
| `P1` | persistência mínima de 1 janela |
| `TRAIN_M2S` | limiar causal baseado em média + 2 desvios no treino |

O conjunto de features utilizado foi o `temporal_core`, composto por 14 variáveis:

```text
fail_rate
n_events
n_failed
n_machines
n_collections
event_FAIL_count
event_LOST_count
lag_1
lag_2
lag_3
rolling_mean_1h
rolling_std_1h
pct_change
zscore_expanding
```

A escolha é metodologicamente adequada porque `temporal_core` é o conjunto de features associado ao modelo vencedor do NB11, permitindo comparação direta entre a LSTM ajustada e a Regressão Logística consolidada no pipeline.

## 11. Configuração Experimental do Tuning

O grid compacto avaliado foi:

| Hiperparâmetro | Valores |
|---|---|
| `sequence_length` | 12, 24, 36 |
| `units` | 8, 16, 32 |
| `dropout` | 0,10; 0,20; 0,30 |
| `learning_rate` | 0,001; 0,0005 |

A configuração geral do treinamento foi:

| Parâmetro | Valor |
|---|---:|
| Folds no `TimeSeriesSplit` | 5 |
| Épocas máximas | 50 |
| `batch_size` | 128 |
| `early_stopping_patience` | 8 |
| `reduce_lr_patience` | 4 |
| Limiar de classificação | 0,50 |
| Protocolo de seleção | F1 médio no `TimeSeriesSplit` |
| Gatilho para uso no NB14 | ganho mínimo de +0,0300 sobre o NB11 |

A execução foi estruturada em duas fases:

### Stage 1 — Triagem

```text
54 configurações × 1 semente × 5 folds = 270 avaliações
```

### Stage 2 — Confirmação

```text
3 melhores configurações × 3 sementes × 5 folds = 45 avaliações
```

### Corte fixo complementar

```text
melhor configuração confirmada × 3 sementes = 3 avaliações
```

Essa estratégia é defensável porque reduz custo computacional, preserva rastreabilidade e evita selecionar a configuração final apenas por uma única inicialização aleatória.

## 12. Diagnóstico por Comprimento de Sequência

O número de sequências elegíveis foi praticamente estável entre os comprimentos avaliados:

| `sequence_length` | Sequências | Positivos | Taxa positiva | Nº de features |
|---:|---:|---:|---:|---:|
| 12 | 6.954 | 2.605 | 0,3746 | 14 |
| 24 | 6.954 | 2.605 | 0,3746 | 14 |
| 36 | 6.944 | 2.605 | 0,3751 | 14 |

A pequena redução em `sequence_length=36` é esperada, pois sequências mais longas exigem histórico anterior suficiente para formar cada instância. A diferença é pequena e não compromete a comparação.

## 13. Stage 1 — Triagem do Grid

O Stage 1 avaliou 54 configurações distintas em 5 folds cada, totalizando 270 avaliações válidas.

As dez melhores configurações por F1 médio foram:

| Rank | `sequence_length` | `units` | `dropout` | `learning_rate` | F1 médio | σ F1 | Recall | Precisão | ROC-AUC | AP | Brier |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 24 | 8 | 0,10 | 0,0010 | 0,5658 | 0,2953 | 0,7495 | 0,4739 | 0,6230 | 0,5408 | 0,2390 |
| 2 | 36 | 8 | 0,10 | 0,0010 | 0,5653 | 0,2933 | 0,7784 | 0,4673 | 0,6327 | 0,5527 | 0,2385 |
| 3 | 24 | 8 | 0,30 | 0,0005 | 0,5643 | 0,2957 | 0,7503 | 0,4719 | 0,6246 | 0,5455 | 0,2374 |
| 4 | 24 | 8 | 0,10 | 0,0005 | 0,5642 | 0,2966 | 0,7523 | 0,4721 | 0,6243 | 0,5399 | 0,2387 |
| 5 | 24 | 8 | 0,30 | 0,0010 | 0,5639 | 0,2970 | 0,7546 | 0,4718 | 0,6249 | 0,5430 | 0,2383 |
| 6 | 12 | 8 | 0,10 | 0,0010 | 0,5638 | 0,2930 | 0,7228 | 0,4741 | 0,6266 | 0,5453 | 0,2326 |
| 7 | 24 | 8 | 0,20 | 0,0010 | 0,5635 | 0,2976 | 0,7375 | 0,4793 | 0,6221 | 0,5378 | 0,2402 |
| 8 | 12 | 8 | 0,10 | 0,0005 | 0,5632 | 0,2919 | 0,7252 | 0,4726 | 0,6258 | 0,5449 | 0,2330 |
| 9 | 12 | 8 | 0,20 | 0,0005 | 0,5631 | 0,2906 | 0,7354 | 0,4693 | 0,6277 | 0,5471 | 0,2325 |
| 10 | 24 | 8 | 0,20 | 0,0005 | 0,5627 | 0,2965 | 0,7503 | 0,4706 | 0,6239 | 0,5419 | 0,2384 |

A triagem indicou que a região mais promissora do grid era formada por modelos pequenos, especialmente com `units=8`. Aumentar a capacidade recorrente para 16 ou 32 unidades não trouxe vantagem clara.

## 14. Padrões Observados no Stage 1

### 14.1 Efeito de `sequence_length`

| `sequence_length` | F1 médio médio | Melhor F1 | Recall médio | Melhor Recall | ROC-AUC médio | Melhor ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|
| 12 | 0,5411 | 0,5638 | 0,6837 | 0,7398 | 0,6126 | 0,6300 |
| 24 | 0,5376 | 0,5658 | 0,6787 | 0,7546 | 0,6114 | 0,6260 |
| 36 | 0,5340 | 0,5653 | 0,6939 | 0,7784 | 0,6018 | 0,6327 |

O melhor F1 individual ocorreu em `sequence_length=24`, mas `sequence_length=36` apresentou maior recall e maior ROC-AUC entre as configurações competitivas. Isso sugere que contexto temporal mais longo pode melhorar sensibilidade ou ordenação relativa, mas não necessariamente melhora o F1 no limiar fixo de 0,50.

### 14.2 Efeito de `units`

| `units` | F1 médio médio | Melhor F1 | Recall médio | Melhor Recall | ROC-AUC médio | Melhor ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|
| 8 | 0,5608 | 0,5658 | 0,7458 | 0,7784 | 0,6228 | 0,6327 |
| 16 | 0,5105 | 0,5180 | 0,5980 | 0,6114 | 0,5838 | 0,5901 |
| 32 | 0,5414 | 0,5508 | 0,7126 | 0,7390 | 0,6191 | 0,6300 |

Esse é um dos achados mais relevantes do NB13a: maior capacidade recorrente não implicou melhor desempenho. O melhor grupo foi `units=8`, reforçando que modelos mais compactos são mais adequados nesta base e nesta formulação.

### 14.3 Efeito de `dropout`

| `dropout` | F1 médio médio | Melhor F1 | Recall médio | Melhor Recall | ROC-AUC médio | Melhor ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|
| 0,10 | 0,5383 | 0,5658 | 0,6857 | 0,7784 | 0,6085 | 0,6327 |
| 0,20 | 0,5368 | 0,5635 | 0,6835 | 0,7627 | 0,6076 | 0,6313 |
| 0,30 | 0,5376 | 0,5643 | 0,6871 | 0,7689 | 0,6096 | 0,6304 |

O dropout não apresentou efeito dominante no Stage 1. As diferenças médias são pequenas diante da variabilidade temporal entre folds.

### 14.4 Efeito de `learning_rate`

| `learning_rate` | F1 médio médio | Melhor F1 | Recall médio | Melhor Recall | ROC-AUC médio | Melhor ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|
| 0,0005 | 0,5354 | 0,5643 | 0,6849 | 0,7628 | 0,6059 | 0,6277 |
| 0,0010 | 0,5398 | 0,5658 | 0,6860 | 0,7784 | 0,6113 | 0,6327 |

O `learning_rate=0,0010` teve leve vantagem média no Stage 1, mas a melhor configuração confirmada no Stage 2 acabou usando `learning_rate=0,0005`, mostrando que a confirmação com múltiplas sementes alterou a leitura inicial.

## 15. Stage 2 — Confirmação com Múltiplas Sementes

As três melhores configurações do Stage 1 foram reavaliadas com três sementes (`42`, `123`, `2026`), totalizando 45 avaliações válidas.

O resultado consolidado do Stage 2 foi:

| Rank | `sequence_length` | `units` | `dropout` | `learning_rate` | F1 médio | σ F1 | Recall | Precisão | ROC-AUC | AP | Brier |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 24 | 8 | 0,30 | 0,0005 | 0,5511 | 0,2823 | 0,7092 | 0,4731 | 0,6324 | 0,5593 | 0,2268 |
| 2 | 36 | 8 | 0,10 | 0,0010 | 0,5448 | 0,2831 | 0,7124 | 0,4720 | 0,6192 | 0,5413 | 0,2301 |
| 3 | 24 | 8 | 0,10 | 0,0010 | 0,5425 | 0,2820 | 0,6850 | 0,4745 | 0,6200 | 0,5450 | 0,2286 |

A melhor configuração confirmada foi:

```text
sequence_length = 24
units = 8
dropout = 0,30
learning_rate = 0,0005
```

Esse resultado é importante porque mostra que a melhor configuração da triagem com uma semente não foi a melhor na confirmação com múltiplas sementes. A configuração `dropout=0,30` e `learning_rate=0,0005`, que havia ficado em terceiro lugar no Stage 1, tornou-se a melhor no Stage 2.

## 16. Melhor Configuração Confirmada

A melhor configuração confirmada apresentou os seguintes resultados no `TimeSeriesSplit`:

| Métrica | Valor |
|---|---:|
| Acurácia média | 0,6260 |
| Desvio-padrão da acurácia | 0,1089 |
| Precisão média | 0,4731 |
| Desvio-padrão da precisão | 0,2732 |
| Recall médio | 0,7092 |
| Desvio-padrão do recall | 0,2669 |
| F1 médio | 0,5511 |
| Desvio-padrão do F1 | 0,2823 |
| ROC-AUC médio | 0,6324 |
| Desvio-padrão do ROC-AUC | 0,1466 |
| Average Precision médio | 0,5593 |
| Desvio-padrão do AP | 0,2821 |
| Brier médio | 0,2268 |
| Desvio-padrão do Brier | 0,0404 |
| Épocas médias executadas | 30,33 |
| Tempo médio de ajuste | 12,69 s |
| Tempo médio total | 13,21 s |

A configuração apresenta desempenho competitivo, mas não supera o NB13 nem o NB11 na métrica principal F1 médio TSCV.

## 17. Análise por Semente da Melhor Configuração

A melhor configuração confirmada apresentou os seguintes desempenhos médios por semente:

| Seed | F1 médio | Recall | Precisão | ROC-AUC | AP | Brier | Épocas médias |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 42 | 0,5643 | 0,7503 | 0,4719 | 0,6246 | 0,5455 | 0,2374 | 35,8 |
| 123 | 0,5417 | 0,6530 | 0,4835 | 0,6310 | 0,5598 | 0,2166 | 33,4 |
| 2026 | 0,5472 | 0,7243 | 0,4640 | 0,6416 | 0,5725 | 0,2264 | 21,8 |

Há variação entre sementes, mas a diferença principal do resultado não vem apenas da semente. A variabilidade temporal entre folds continua sendo o fator dominante.

## 18. Análise por Fold da Melhor Configuração

A média por fold da melhor configuração confirmada foi:

| Fold | Taxa positiva | F1 médio | Recall | Precisão | ROC-AUC | AP | Brier |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0,2071 | 0,1660 | 0,2417 | 0,1349 | 0,3574 | 0,1641 | 0,2706 |
| 2 | 0,1432 | 0,2975 | 0,6767 | 0,1911 | 0,6570 | 0,3204 | 0,2535 |
| 3 | 0,4461 | 0,6433 | 0,7756 | 0,5515 | 0,6895 | 0,6498 | 0,2405 |
| 4 | 0,6842 | 0,8205 | 0,9437 | 0,7267 | 0,7071 | 0,8169 | 0,1922 |
| 5 | 0,6868 | 0,8282 | 0,9083 | 0,7614 | 0,7510 | 0,8453 | 0,1773 |

O comportamento por fold confirma novamente o padrão observado no NB11, NB12 e NB13: os folds 1 e 2 são substancialmente mais difíceis, enquanto os folds 3 a 5 apresentam desempenho alto.

Esse padrão indica instabilidade temporal estrutural da série, não uma falha específica do tuning da LSTM.

## 19. Resultado no Corte Fixo 80/20

O corte fixo 80/20 foi executado apenas como análise complementar da melhor configuração confirmada.

O resultado agregado foi:

| Métrica | Valor |
|---|---:|
| F1 médio | 0,8505 |
| Desvio-padrão do F1 | 0,0018 |
| Recall médio | 0,9282 |
| Precisão média | 0,7851 |
| ROC-AUC médio | 0,7583 |
| Average Precision médio | 0,8668 |
| Brier médio | 0,1687 |
| Épocas médias | 19,0 |

Esse resultado é alto, mas deve permanecer como análise complementar. A própria cadeia NB11–NB13 já mostrou que o corte fixo é favorecido pela concentração de positivos no conjunto de teste. A evidência principal continua sendo o `TimeSeriesSplit`.

## 20. Comparação com o NB13

A melhor LSTM do NB13 havia apresentado:

| Origem | Configuração | F1 TSCV | Recall | Precisão | ROC-AUC | AP | Brier |
|---|---|---:|---:|---:|---:|---:|---:|
| NB13 | L=24, units=8, dropout=0,20, LR=0,001 | 0,5580 | 0,7296 | 0,4755 | 0,6185 | 0,5484 | 0,2398 |
| NB13a | L=24, units=8, dropout=0,30, LR=0,0005 | 0,5511 | 0,7092 | 0,4731 | 0,6324 | 0,5593 | 0,2268 |

A comparação mostra um resultado misto:

| Métrica | Delta NB13a − NB13 | Interpretação |
|---|---:|---|
| F1 | −0,0069 | Piorou levemente |
| Recall | −0,0204 | Piorou levemente |
| Precisão | −0,0024 | Praticamente igual |
| ROC-AUC | +0,0139 | Melhorou |
| Average Precision | +0,0109 | Melhorou |
| Brier | −0,0130 | Melhorou |

Portanto, o tuning compacto **não melhorou a métrica de seleção principal**, que é o F1 médio TSCV. Entretanto, melhorou métricas de ordenação e qualidade probabilística relativa, como ROC-AUC, Average Precision e Brier.

Esse achado deve ser interpretado com cautela: embora a LSTM ajustada tenha melhorado algumas métricas, ela não melhorou o critério decisório principal do pipeline.

## 21. Comparação com o NB11

A comparação decisiva é contra o modelo tabular vencedor do NB11:

| Modelo | F1 TSCV | Recall | Precisão | ROC-AUC | Brier |
|---|---:|---:|---:|---:|---:|
| NB11 — LogReg / `temporal_core` | 0,5774 | — | — | — | — |
| NB13 — Melhor LSTM | 0,5580 | 0,7296 | 0,4755 | 0,6185 | 0,2398 |
| NB13a — LSTM ajustada | 0,5511 | 0,7092 | 0,4731 | 0,6324 | 0,2268 |

Em termos de F1, o NB13a ficou:

```text
Delta NB13a − NB11 = −0,0263
Delta NB13a − NB13 = −0,0069
```

Como o gatilho definido para substituir os escores do NB11 no NB14 exigia:

```text
Delta mínimo = +0,0300
```

a LSTM ajustada permanece abaixo do critério de substituição.

Para deslocar o NB11, seria necessário atingir aproximadamente:

```text
0,5774 + 0,0300 = 0,6074
```

O NB13a obteve:

```text
0,5511
```

Logo, o tuning compacto não altera a decisão de pipeline.

## 22. O Que Melhorou com o NB13a

Mesmo sem superar o NB11, o NB13a melhorou a dissertação em vários aspectos.

### 22.1 Responde a uma crítica provável da banca

Antes do NB13a, a banca poderia perguntar:

> A LSTM ficou próxima da Regressão Logística. E se houvesse tuning de hiperparâmetros?

Agora a resposta é empiricamente sustentada:

> Foi realizado tuning compacto e controlado, com variação de `sequence_length`, `units`, `dropout` e `learning_rate`. A LSTM ajustada não superou o modelo tabular vencedor pelo critério definido.

### 22.2 Fortalece o argumento de parcimônia

O tuning demonstrou que aumentar ou ajustar moderadamente a complexidade da LSTM não deslocou a Regressão Logística. Isso reforça que o modelo tabular permanece preferível por simplicidade, interpretabilidade e desempenho.

### 22.3 Mostra que a configuração original do NB13 era competitiva

A melhor configuração do NB13 (`L=24`, `units=8`, `dropout=0,20`, `LR=0,001`) ficou muito próxima das melhores configurações encontradas no tuning. Isso mostra que o NB13 não estava em uma região ruim do espaço de hiperparâmetros.

### 22.4 Identifica padrões úteis do espaço de busca

O NB13a mostrou que:

1. `units=8` foi consistentemente melhor que 16 e 32.
2. `sequence_length=24` permaneceu competitivo.
3. `sequence_length=36` melhorou recall e ROC-AUC em algumas configurações, mas não F1.
4. O tuning melhora algumas métricas auxiliares, mas não a métrica decisória principal.
5. A instabilidade temporal entre folds permanece dominante.

### 22.5 Melhora a maturidade metodológica

A existência do NB13a permite dizer que a manutenção do NB11 não decorre de ausência de tentativa com modelos sequenciais, mas de evidência experimental acumulada.

## 23. O Que Não Melhorou

O NB13a também mostra limites importantes.

### 23.1 O F1 médio não melhorou

O melhor F1 do NB13 era 0,5580. O melhor F1 confirmado no NB13a foi 0,5511. Portanto, a confirmação com múltiplas sementes reduziu a expectativa inicial observada no Stage 1.

### 23.2 O recall caiu

O recall médio da melhor LSTM caiu de 0,7296 no NB13 para 0,7092 no NB13a. Como recall é relevante para antecipação de falhas, essa queda reforça que a configuração ajustada não deve substituir o NB13 nem o NB11.

### 23.3 O ganho de ROC-AUC e Brier não compensa a perda em F1

O ROC-AUC subiu de 0,6185 para 0,6324, e o Brier melhorou de 0,2398 para 0,2268. Esses ganhos sugerem melhor ordenação relativa e melhor qualidade média dos escores, mas a decisão do pipeline estava baseada em F1 médio TSCV. Portanto, não são suficientes para alterar o fluxo NB14.

### 23.4 A instabilidade temporal permaneceu

Os folds 1 e 2 continuam com F1 baixo. O tuning não resolveu a não-estacionaridade da série. Isso reforça que a instabilidade observada é estrutural, não apenas consequência de hiperparâmetros subótimos.

## 24. Interpretação Metodológica

O NB13a deve ser interpretado como uma etapa de robustez metodológica.

Ele não mostra que LSTM é inadequada em sentido geral. Mostra apenas que, nesta formulação, com estes dados, este cenário principal, este conjunto de features e este grid compacto, a LSTM ajustada não superou o modelo tabular vencedor.

A formulação mais segura é:

```text
Nas condições avaliadas, o tuning compacto da LSTM não alterou a conclusão do NB13.
A arquitetura recorrente permaneceu competitiva, mas não apresentou ganho suficiente
para deslocar a Regressão Logística com temporal_core como referência principal do pipeline.
```

Isso evita uma conclusão excessiva contra redes neurais recorrentes e preserva rigor científico.

## 25. Relação com a Decisão do NB14

A decisão do NB14 permanece:

```text
keep_nb11_scores_for_nb14
```

Portanto:

1. o NB14 deve continuar consumindo os escores do modelo vencedor do NB11;
2. a LSTM permanece documentada como comparação sequencial não trivial;
3. o NB13a registra que a conclusão do NB13 foi testada por tuning compacto;
4. o NB13a não altera o cenário principal;
5. o NB13a não desloca a análise decisória baseada no NB11.

A decisão é metodologicamente sólida, pois foi testada contra:

| Comparação | Resultado |
|---|---|
| EWMA causal | LSTM supera |
| Melhor baseline simples | LSTM supera marginalmente |
| LSTM NB13 sem tuning ampliado | NB13a não supera |
| LogReg NB11 | NB13a não supera |
| Gatilho de +0,0300 | não atingido |

## 26. Limitações do NB13a

As limitações principais são:

1. O tuning é compacto, não exaustivo.
2. O grid não inclui múltiplas camadas LSTM.
3. O grid não inclui GRU, TCN, Transformer ou modelos sequenciais alternativos.
4. O grid não testa diferentes funções de perda.
5. O limiar de classificação permanece fixo em 0,50.
6. A seleção é baseada em F1 médio TSCV, não em custo decisório direto.
7. O tuning foi restrito ao cenário principal e ao `temporal_core`.
8. O corte fixo é apenas complementar.
9. A instabilidade temporal entre folds permanece elevada.
10. A melhora em métricas auxiliares não foi usada para alterar a decisão principal.

Essas limitações não invalidam o NB13a. Elas delimitam corretamente seu papel: análise complementar de robustez, não busca neural ampla.

## 27. Valor Argumentativo para a Dissertação

O NB13a é útil para a dissertação porque fortalece a resposta a quatro perguntas de banca:

### 27.1 Por que não usar LSTM como modelo principal?

Porque a LSTM foi avaliada no NB13 e ajustada no NB13a por tuning compacto. Mesmo assim, não superou o modelo tabular vencedor pelo critério previamente definido.

### 27.2 A LSTM ficou próxima. Não faltou tuning?

Foi executado tuning compacto variando `sequence_length`, `units`, `dropout` e `learning_rate`. A melhor configuração confirmada ficou abaixo tanto do NB11 quanto da melhor LSTM anterior em F1 médio TSCV.

### 27.3 O modelo tabular é uma escolha simplista?

Não. O modelo tabular permanece preferencial após comparação com baselines temporais, LSTM simples e LSTM ajustada. A escolha é baseada em parcimônia, interpretabilidade e desempenho sob validação temporal.

### 27.4 O problema está na arquitetura ou na série?

Os resultados sugerem que a limitação principal está na instabilidade temporal da série e na formulação do problema, não apenas na arquitetura. A LSTM melhora algumas métricas auxiliares, mas não resolve a variabilidade dos folds iniciais.

## 28. Recomendações para Texto da Dissertação

A formulação recomendada para a dissertação é:

```text
Como análise complementar, foi realizado um tuning compacto da arquitetura LSTM no cenário principal e no conjunto de features temporal_core. O grid avaliou diferentes comprimentos de sequência, tamanhos da camada recorrente, taxas de dropout e learning rates, em validação temporal por TimeSeriesSplit. A melhor configuração confirmada obteve F1 médio de 0,5511, valor inferior ao da melhor LSTM do NB13 (0,5580) e ao modelo tabular vencedor do NB11 (0,5774). Assim, o tuning compacto não alterou a decisão metodológica de manter os escores do NB11 como entrada preferencial da análise decisória do NB14. O resultado reforça a opção por um modelo tabular interpretável, sem descartar a investigação futura de arquiteturas sequenciais mais amplas.
```

Essa redação é segura, transparente e não fragiliza o trabalho.

## 29. Encaminhamento para os Próximos Notebooks

O próximo passo pode ser avançar para o NB14 mantendo a cadeia principal definida no NB11.

O encaminhamento recomendado é:

| Notebook | Decisão |
|---|---|
| NB13 | Manter como comparação sequencial não trivial |
| NB13a | Registrar como tuning compacto que não alterou a decisão |
| NB14 | Usar escores do NB11 |
| NB15 | Visualizar decisão estabilizada após NB14 |
| NB16 | Registrar inventário e síntese NB10–NB15, incluindo NB13a |

O NB13a deve ser citado no NB16 como notebook intermediário de robustez metodológica da LSTM, sem abrir nova cadeia de experimentos.

## 30. Conclusão da Etapa

O NB13a cumpriu seu papel como notebook intermediário de tuning compacto da LSTM.

A execução confirmou vinte pontos centrais:

1. O NB13a respeitou a regra de notebook intermediário.
2. Nenhum artefato do NB13 foi sobrescrito.
3. Todos os outputs usam prefixo `13a_`.
4. O tuning foi restrito ao cenário principal `W5_K24_H12_P1_TRAIN_M2S`.
5. O tuning foi restrito ao feature set `temporal_core`.
6. Foram avaliados três comprimentos de sequência: 12, 24 e 36.
7. Foram avaliadas três capacidades recorrentes: 8, 16 e 32 unidades.
8. Foram avaliados três valores de dropout: 0,10; 0,20; 0,30.
9. Foram avaliados dois learning rates: 0,001 e 0,0005.
10. O Stage 1 avaliou 54 configurações e 270 folds válidos.
11. O Stage 2 confirmou as três melhores configurações com três sementes.
12. O Stage 2 totalizou 45 avaliações válidas.
13. O corte fixo complementar foi executado com três sementes.
14. A melhor configuração confirmada foi `sequence_length=24`, `units=8`, `dropout=0,30`, `learning_rate=0,0005`.
15. O melhor F1 médio TSCV confirmado foi 0,5511.
16. O F1 da LSTM ajustada ficou abaixo do NB13, que havia obtido 0,5580.
17. O F1 da LSTM ajustada ficou abaixo do NB11, que havia obtido 0,5774.
18. O delta contra o NB11 foi −0,0263, não atingindo o gatilho de +0,0300.
19. A decisão `keep_nb11_scores_for_nb14` permanece válida.
20. O NB13a fortalece a narrativa de parcimônia, interpretabilidade e robustez metodológica da escolha pelo modelo tabular.

Conclusão central do NB13a: o tuning compacto da LSTM não melhorou a métrica decisória principal e não alterou a decisão do pipeline. A Regressão Logística com `temporal_core`, consolidada no NB11, permanece como referência preferencial para o NB14. A LSTM ajustada deve ser registrada como comparação sequencial complementar, útil para a defesa metodológica, mas sem substituir os escores do modelo tabular na análise decisória.
